# 00 · Setup and verify

Welcome! This workshop series teaches **Flyte v2 / Union 2.x** by building one system,
end to end, across nine chapters:

> **Review Radar** — a product-review intelligence pipeline. We'll take raw customer
> reviews, clean and enrich them at scale, engineer features reproducibly, score the whole
> corpus on warm containers, train a sentiment model, serve it as a live API, and finish
> with an agent that triages new reviews using that model.

```
raw reviews → data flow → processing at scale → caching → batch scoring
                                        ↘ training → serving → agents
```

Each chapter stands alone (you can run any notebook independently), but together they tell
the story of taking a workload from first ingest to production serving on Union.

**This chapter's goals**

1. Connect the SDK to the Union deployment for this engagement (BYOC or self-managed)
2. Run a first task remotely, straight from a notebook cell
3. Understand *how* notebook code executes remotely (and the four authoring rules)
4. Verify the image build → registry → run path works end to end

**Prerequisites** (see [README](./README.md) and
[appendix A](./appendix/A-deployment-adaptation.md)): Python **3.12** kernel · a Union
endpoint + credentials · a project/domain for the workshop.

## 1. Install driver-side dependencies

This workshop standardizes on [**uv**](https://docs.astral.sh/uv/). Your *laptop*
dependencies are pinned in `pyproject.toml` and locked in `uv.lock`. Task images declare
their own dependencies separately (you'll see how in section 5).

If you followed the [README](./README.md) setup (`uv sync` + `uv run jupyter lab`),
everything is already installed and you can skip the next cell — run it to (re)install
into the current kernel otherwise.

In [1]:
# Installs the pinned deps into THIS kernel's interpreter (idempotent).
# Prefer the README setup instead: `uv sync && uv run jupyter lab`.
import sys

!uv pip install -q --python {sys.executable} -r pyproject.toml

In [2]:
!flyte --version

Flyte SDK version: 2.5.7


## 2. Connect to the control plane

Copy [`config-templates/config.yaml.example`](./config-templates/config.yaml.example) to
`~/.flyte/config.yaml` and set the endpoint for this engagement, or generate one:

```bash
flyte create config --endpoint dns:///<union-endpoint>
```

Every notebook in this series then connects with a single line: `flyte.init_from_config()`.
The first call opens a browser window to authenticate against the deployment's IdP.

In [1]:
import flyte

flyte.init_from_config()

# Alternative, explicit form (useful when testing a second cluster or in CI):
# flyte.init(
#     endpoint="dns:///<union-endpoint>",
#     org="<org-name>",
#     project="onboarding",
#     domain="development",
# )

In [4]:
# Client-side workshop settings (.env) — project, domain, registry, object store.
# NOTE: this module is CLIENT-SIDE ONLY; never import it inside a task body.
from workshop_config import preflight

WS = preflight()

project=flytesnacks domain=development
registry=us-central1-docker.pkg.dev/flytetf8/selfmanaged object_store=gs://selfmanaged-gcs-testing


In [2]:
# Confirm the control plane answers and your project exists
!flyte get project

                                    Projects                                    
┌────────────────────────────────┬──────────┬───────────┬──────────┬───────────┐
│                                │          │ Descripti │          │           │
│ Name                           │ Id       │ on        │ State    │ Labels    │
╞════════════════════════════════╪══════════╪═══════════╪══════════╪═══════════╡
│ flytesnacks                    │ flytesna │ Some      │ PROJECT_ │ test:     │
│                                │ cks      │ descripti │ STATE_AC │ test-labe │
│                                │          │ on for    │ TIVE     │ l         │
│                                │          │ the       │          │           │
│                                │          │ project   │          │           │
│ flytetester                    │ flytetes │ flytetest │ PROJECT_ │           │
│                                │ ter      │ er        │ STATE_AC │           │
│                           

## 3. How notebook code runs remotely

When you call `flyte.run` from a notebook, the SDK detects the interactive session and
ships a **pickled code bundle** of your cell-defined tasks to the cluster — no `.py` file
or git checkout required. That convenience comes with four **authoring rules**:

1. **Define tasks and their helpers in notebook cells** (or installed packages). Functions
   imported from local modules (like `workshop_config.py`) are pickled *by reference* and
   will fail remotely with `ModuleNotFoundError`. Pass config values into tasks as inputs.
2. **The kernel's Python minor version must match the task image** — everything here pins 3.12.
3. **`flyte.deploy()` is not supported from interactive sessions.** Anything that needs a
   *deployment* (triggers/schedules, tasks referenced via `flyte.remote.Task.get`,
   connectors, apps) lives in [`scripts/`](./scripts/) and is driven with
   `!flyte deploy ...` / `!python scripts/...` shell cells.
4. **`Environment(include=...)` (bundling extra files) is not supported in interactive
   runs** — that's also a `scripts/` job.

Everything else — `TaskEnvironment`, fan-out, caching, reusable containers, Ray — works
exactly the same from a notebook as from a file.

## 4. Hello, remote world

A `TaskEnvironment` groups tasks that share an image, resources, and runtime settings.
The default image (`flyte.Image.from_debian_base()`) ships with the `flyte` SDK
pre-installed, so this first run needs no image build.

In [3]:
env = flyte.TaskEnvironment(
    name="hello",
    resources=flyte.Resources(cpu="1", memory="512Mi"),
)


@env.task
async def greet(name: str) -> str:
    return f"Review Radar reporting for duty, {name}!"

In [6]:
run = flyte.run(greet, name="team")
run.wait()  # wait for the run to complete
print(run.url)   # open this in the Union UI
run.outputs()

> Building 1 image...

> Building image flyte for environment hello

✓ Built image for environment hello: ghcr.io/flyteorg/flyte:py3.12-v2.5.11

Run 'up5cnqq5f4vqsjc7zxjb' completed successfully.

https://demo.hosted.unionai.cloud/v2/domain/development/project/davide/runs/up5cnqq5f4vqsjc7zxjb


ActionOutputs(o0="Review Radar reporting for duty, team!")

### Under the hood

That run executed in a container in the workshop project/domain. If this deployment is
BYOC or self-managed (you operate the data plane), you can see it as a pod in the
`<project>-<domain>` namespace:

```bash
kubectl get pods -n onboarding-development
```

## 5. Verify the image build path

Real workloads need dependencies, so images get built. The **remote Image Builder** builds
in-cluster (or in Union's managed builder, depending on deployment) and pushes to the
registry configured for this deployment (`IMAGE_REGISTRY` in `.env`, or Union-managed).
This cell verifies the whole path: build → push → pull → run.

The first execution takes a few minutes (image build); re-running is instant because image
builds are content-addressed and cached.

In [ ]:
image = (
    flyte.Image.from_debian_base(name="workshop-smoke", python_version=(3, 12))
    .with_pip_packages("pyjokes==0.8.3")
)

smoke_env = flyte.TaskEnvironment(
    name="image_smoke",
    image=image,
    resources=flyte.Resources(cpu="1", memory="512Mi"),
)


@smoke_env.task
async def tell_joke() -> str:
    import pyjokes
    return pyjokes.get_joke()


run = flyte.run(tell_joke)
print(run.url)
run.wait()
print(run.outputs())

> **If the build fails** with a push/pull permission error, the builder or the task
> identity is missing write/read access on the registry — see
> [appendix A](./appendix/A-deployment-adaptation.md) → *Image registry*.

## Further reading

- Union docs → User guide → Task programming → Notebooks
- Next: [01-authoring-fundamentals](./01-authoring-fundamentals.ipynb) — where the Review
  Radar story begins with its first pipeline